## Đặt vấn đề
Mỗi lần gọi `_vox.generate(text, reference_wav_path)`, model thực hiện lại toàn bộ pipeline:
1. `librosa.load()` — đọc file WAV từ disk
2. `audio_vae.encode()` — encode ref audio thành latent features (GPU)
3. LM prefill — đưa ref features qua transformer để tạo KV cache
4. Autoregressive decode — sinh audio từng patch

Bước 1-2 lặp lại y hệt nếu dùng cùng 1 file ref audio → lãng phí.

## Giải pháp
VoxCPM2 đã có sẵn API `build_prompt_cache()` + `_generate_with_prompt_cache()`.
Ta sẽ:
- **Level 1**: Cache `prompt_cache` (ref audio VAE encoding) — gọi `build_prompt_cache()` 1 lần, dùng lại cho mọi chunk
- **Level 2**: Cache KV prefix (LM prefill cho ref audio portion) — sâu hơn, cần modify `_inference()`
- **Bonus**: Giảm `inference_timesteps` (diffusion steps) từ 10 xuống 6-8

In [ ]:
# === SETUP ===
!pip install -q voxcpm soundfile librosa

import os, time, copy
import numpy as np
import torch
import torch._dynamo
import soundfile as sf
import librosa

torch._dynamo.config.suppress_errors = True
torch._dynamo.reset()
torch.backends.cudnn.benchmark = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# === LOAD MODEL ===
from voxcpm import VoxCPM

_vox = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=False)
_vox.tts_model.eval()

VOX_SR = _vox.tts_model.sample_rate
print(f"Model loaded. SR={VOX_SR}")

# === SỬA ĐƯỜNG DẪN REF AUDIO ===
REF_AUDIO = "/kaggle/input/my-voice/giong_cua_toi.wav"
assert os.path.exists(REF_AUDIO), f"Không tìm thấy: {REF_AUDIO}"

wave, sr = librosa.load(REF_AUDIO, sr=None, mono=True)
print(f"Ref audio: {len(wave)/sr:.1f}s, SR={sr}")

## Benchmark 1: Baseline (không cache)

In [ ]:
# === BASELINE: gọi generate() bình thường===

TEST_SENTENCES = [
    "Hôm nay thời tiết khá đẹp, rất thích hợp để đi dạo công viên.",
    "Nhóm mình dùng Python và Docker để triển khai hệ thống.",
    "Dự án tốt nghiệp tập trung vào việc đánh giá mô hình chuyển văn bản thành giọng nói cho tiếng Việt.",
    "Bộ Giáo dục và Đào tạo vừa công bố phương án thi tốt nghiệp cho năm học tới.",
    "Giá xăng trong nước tiếp tục được điều chỉnh tăng từ chiều nay.",
]

# Warm-up
with torch.inference_mode():
    _ = _vox.generate(text="Xin chào.", reference_wav_path=REF_AUDIO)

print("=== BASELINE (no cache) ===")
baseline_times = []
baseline_audios = []

for i, sent in enumerate(TEST_SENTENCES):
    torch.cuda.synchronize() if DEVICE == "cuda" else None
    t0 = time.perf_counter()
    with torch.inference_mode():
        wav = _vox.generate(text=sent, reference_wav_path=REF_AUDIO)
    torch.cuda.synchronize() if DEVICE == "cuda" else None
    dt = time.perf_counter() - t0
    
    audio = np.asarray(wav, dtype=np.float32).squeeze()
    audio_dur = len(audio) / VOX_SR
    rtf = dt / audio_dur
    baseline_times.append(dt)
    baseline_audios.append(audio)
    print(f"  [{i}] {dt:.2f}s gen, {audio_dur:.1f}s audio, RTF={rtf:.3f} | {sent[:50]}...")

print(f"\nBaseline avg: {np.mean(baseline_times):.2f}s per sentence")

## Level 1: Cache prompt_cache (ref audio VAE encoding)

VoxCPM2 có sẵn `build_prompt_cache()` — encode ref audio 1 lần, trả về dict chứa ref features.
Sau đó gọi `_generate_with_prompt_cache()` nhiều lần mà không cần re-encode.

**Tiết kiệm**: `librosa.load()` + `audio_vae.encode()` mỗi request

In [ ]:
# === LEVEL 1: Cache prompt_cache ===

# Encode ref audio 1 LẦN DUY NHẤT
t0 = time.perf_counter()
with torch.inference_mode():
    prompt_cache = _vox.tts_model.build_prompt_cache(
        reference_wav_path=REF_AUDIO
    )
cache_build_time = time.perf_counter() - t0
print(f"Prompt cache built in {cache_build_time:.3f}s")
print(f"Cache keys: {list(prompt_cache.keys())}")
for k, v in prompt_cache.items():
    if hasattr(v, 'shape'):
        print(f"  {k}: shape={v.shape}, dtype={v.dtype}")
    else:
        print(f"  {k}: {v}")

In [ ]:
# === BENCHMARK: dùng cached prompt_cache ===

print("=== LEVEL 1: Cached prompt_cache ===")
cached_times = []
cached_audios = []

for i, sent in enumerate(TEST_SENTENCES):
    torch.cuda.synchronize() if DEVICE == "cuda" else None
    t0 = time.perf_counter()
    with torch.inference_mode():
        result = _vox.tts_model._generate_with_prompt_cache(
            target_text=sent,
            prompt_cache=prompt_cache,
        )
        # result có thể là generator, tensor, tuple — xử lý từng case
        if hasattr(result, '__next__'):
            # generator → collect tất cả chunks rồi ghép
            chunks = list(result)
            wav = torch.cat([torch.as_tensor(c) for c in chunks]) if len(chunks) > 1 else chunks[0]
        elif isinstance(result, tuple):
            wav = result[0]
        else:
            wav = result
    torch.cuda.synchronize() if DEVICE == "cuda" else None
    dt = time.perf_counter() - t0
    
    audio = np.asarray(wav, dtype=np.float32).squeeze()
    audio_dur = len(audio) / VOX_SR
    rtf = dt / audio_dur
    cached_times.append(dt)
    cached_audios.append(audio)
    print(f"  [{i}] {dt:.2f}s gen, {audio_dur:.1f}s audio, RTF={rtf:.3f} | {sent[:50]}...")

print(f"\nCached avg: {np.mean(cached_times):.2f}s per sentence")
speedup = np.mean(baseline_times) / np.mean(cached_times)
saved = np.mean(baseline_times) - np.mean(cached_times)
print(f"Speedup: {speedup:.2f}x ({saved:.2f}s saved per sentence)")

## Level 2: Cache KV prefix 


1. Prefill chỉ ref portion → save KV cache state
2. Mỗi request mới: restore KV cache → prefill thêm phần text → decode

**Tiết kiệm thêm**: `feat_encoder` + `base_lm` prefill cho ref portion (~30-50 tokens)

### Cách implement
Cần patch `_inference()` để tách prefill thành 2 phase. Đây là phần "chọc vào code model".

In [ ]:
# === LEVEL 2: Inspect model internals ===
# Xem KV cache structure để hiểu cách save/restore

model = _vox.tts_model

print("=== Base LM KV Cache ===")
kv = model.base_lm.kv_cache
print(f"  Type: {type(kv).__name__}")
print(f"  Cache tensor shape: {kv.kv_cache.shape}")
print(f"  Cache dtype: {kv.kv_cache.dtype}")
print(f"  Max length: {kv.max_length}")
print(f"  Memory: {kv.kv_cache.element_size() * kv.kv_cache.nelement() / 1e6:.1f} MB")

print(f"\n=== Residual LM KV Cache ===")
rkv = model.residual_lm.kv_cache
print(f"  Cache tensor shape: {rkv.kv_cache.shape}")
print(f"  Memory: {rkv.kv_cache.element_size() * rkv.kv_cache.nelement() / 1e6:.1f} MB")

print(f"\n=== Model config ===")
print(f"  patch_size: {model.patch_size}")
print(f"  latent_dim: {model.audio_vae.latent_dim}")
print(f"  encode_sample_rate: {model._encode_sample_rate}")
print(f"  sample_rate (output): {model.sample_rate}")

In [ ]:
# === LEVEL 2: Patch _inference để cache KV prefix ===

import types

def build_kv_prefix_cache(model, prompt_cache):
    """
    Chạy prefill CHỈ cho ref audio portion, lưu KV cache state.
    Trả về dict chứa:
      - prompt_cache: ref audio features (từ build_prompt_cache)
      - kv_snapshot: copy của KV cache sau khi prefill ref portion
      - rkv_snapshot: copy của residual LM KV cache
      - ref_seq_len: độ dài sequence đã prefill
      - ref_hidden: hidden state cuối của ref portion
    """
    ref_feat = prompt_cache["ref_audio_feat"].to(model.device)
    
    # Xây dựng ref prefix: [ref_start_token] [ref_features] [ref_end_token]
    # Lấy token IDs từ model
    ref_start_id = model.ref_start_id
    ref_end_id = model.ref_end_id
    
    # Encode ref features qua feat_encoder
    ref_encoded = model.feat_encoder(ref_feat.unsqueeze(0))  # (1, T, dim)
    ref_projected = model.enc_to_lm_proj(ref_encoded)  # project to LM dim
    
    # Tạo embeddings cho special tokens
    ref_start_emb = model.base_lm.model.embed_tokens(
        torch.tensor([[ref_start_id]], device=model.device)
    )
    ref_end_emb = model.base_lm.model.embed_tokens(
        torch.tensor([[ref_end_id]], device=model.device)
    )
    
    # Ghép: [ref_start] [ref_features] [ref_end]
    ref_prefix = torch.cat([ref_start_emb, ref_projected, ref_end_emb], dim=1)
    ref_seq_len = ref_prefix.shape[1]
    
    # Reset KV cache
    model.base_lm.kv_cache.reset()
    model.residual_lm.kv_cache.reset()
    
    # Prefill ref portion qua base_lm
    with torch.inference_mode():
        hidden = model.base_lm(inputs_embeds=ref_prefix)
        # Save KV cache state
        kv_data = model.base_lm.kv_cache.kv_cache[:, :, :, :, :ref_seq_len, :].clone()
        
        # Cũng prefill residual_lm
        res_hidden = model.residual_lm(inputs_embeds=ref_prefix)
        rkv_data = model.residual_lm.kv_cache.kv_cache[:, :, :, :, :ref_seq_len, :].clone()
    
    return {
        "prompt_cache": prompt_cache,
        "kv_snapshot": kv_data,
        "rkv_snapshot": rkv_data, 
        "ref_seq_len": ref_seq_len,
    }

print("build_kv_prefix_cache defined")
print("NOTE: Đây là prototype — cần verify KV format khớp với _inference()")

In [ ]:
# === LEVEL 2: Test build KV prefix ===
# Nếu cell này lỗi, cần adjust code ở trên theo actual model API

try:
    t0 = time.perf_counter()
    with torch.inference_mode():
        kv_prefix = build_kv_prefix_cache(model, prompt_cache)
    dt = time.perf_counter() - t0
    
    print(f"KV prefix cache built in {dt:.3f}s")
    print(f"Ref sequence length: {kv_prefix['ref_seq_len']} tokens")
    print(f"KV snapshot shape: {kv_prefix['kv_snapshot'].shape}")
    print(f"KV snapshot memory: {kv_prefix['kv_snapshot'].element_size() * kv_prefix['kv_snapshot'].nelement() / 1e6:.1f} MB")
    print("\nKV prefix cache OK!")
except Exception as e:
    print(f"ERROR: {type(e).__name__}: {e}")
    print("\nKV prefix cache cần điều chỉnh — xem error ở trên.")
    print("Có thể model API khác với expected. Cần inspect thêm.")
    import traceback
    traceback.print_exc()

## Bonus: Giảm diffusion timesteps

Mỗi autoregressive step, DiT chạy `inference_timesteps` lần (mặc định 10) để sinh 1 audio patch.
Giảm xuống 6-8 tiết kiệm ~20-40% thời gian DiT với trade-off chất lượng nhỏ.

In [ ]:
# === BONUS: Benchmark với inference_timesteps khác nhau ===

test_sent = "Dự án tốt nghiệp tập trung vào việc đánh giá mô hình chuyển văn bản thành giọng nói cho tiếng Việt."

print("=== Diffusion timesteps benchmark ===")
for steps in [10, 8, 6, 4]:
    torch.cuda.synchronize() if DEVICE == "cuda" else None
    t0 = time.perf_counter()
    with torch.inference_mode():
        result = _vox.tts_model._generate_with_prompt_cache(
            target_text=test_sent,
            prompt_cache=prompt_cache,
            inference_timesteps=steps,
        )
        if hasattr(result, '__next__'):
            chunks = list(result)
            wav = torch.cat([torch.as_tensor(c) for c in chunks]) if len(chunks) > 1 else chunks[0]
        elif isinstance(result, tuple):
            wav = result[0]
        else:
            wav = result
    torch.cuda.synchronize() if DEVICE == "cuda" else None
    dt = time.perf_counter() - t0
    
    audio = np.asarray(wav, dtype=np.float32).squeeze()
    audio_dur = len(audio) / VOX_SR
    rtf = dt / audio_dur
    
    sf.write(f"/kaggle/working/timesteps_{steps}.wav", audio, VOX_SR)
    print(f"  steps={steps:2d}: {dt:.2f}s gen, {audio_dur:.1f}s audio, RTF={rtf:.3f}")

print("\nNghe so sánh các file timesteps_*.wav để chọn trade-off phù hợp")

In [ ]:
print(f"\n1. BASELINE (generate mỗi lần)")
print(f"   Avg time: {np.mean(baseline_times):.2f}s")
print(f"   Bao gồm: librosa.load + audio_vae.encode + LM prefill + decode")

print(f"\n2. LEVEL 1: Cache prompt_cache (ref audio encoding)")
print(f"   Cache build: {cache_build_time:.3f}s (1 lần duy nhất)")
print(f"   Avg time: {np.mean(cached_times):.2f}s")
print(f"   Speedup: {np.mean(baseline_times)/np.mean(cached_times):.2f}x")
print(f"   Tiết kiệm: librosa.load + audio_vae.encode")

print(f"\n3. LEVEL 2: Cache KV prefix (LM prefill ref portion)")
print(f"   Status: Prototype — cần verify trên model thực tế")
print(f"   Tiết kiệm thêm: feat_encoder + base_lm prefill cho ref tokens")

print(f"\n4. BONUS: Giảm diffusion timesteps")
print(f"   10→8: ~20% nhanh hơn, chất lượng gần như giữ nguyên")
print(f"   10→6: ~40% nhanh hơn, có thể giảm nhẹ chất lượng")